# Week 3 — Data Preprocessing (v2, revised during Week 4)

This notebook prepares sold CRMLS single-family records for modeling and writes `outputs/02_preprocessed_v2.csv`.
The original `outputs/02_preprocessed.csv` is left untouched for reference.

What changed from v1:
- More fields are loaded: `PostalCode` (location), `YearBuilt`, `GarageSpaces`, `ParkingTotal`, `Stories`, `PoolPrivateYN`, and `Latitude`/`Longitude` (kept for the Week 5 tree models).
- A feature-validity rule removes rows with missing or implausible `LivingArea` from every split.
- The training window is re-selected with the actual model (`LinearRegression`), using only months before the test month.
- Scaling, imputation, and encoding are no longer baked into the CSV. They are fit on training rows only inside an sklearn `Pipeline` (see `model_pipeline.py`), so every model uses identical preprocessing.
- The constant `PropertyType`/`PropertySubType` one-hot columns are dropped, because every row is Residential / SingleFamilyResidence.

In [1]:
from pathlib import Path

import pandas as pd

import model_pipeline as mp

OUTPUT_FILE = Path('outputs/02_preprocessed_v2.csv')
TEST_MONTH = pd.Period('2026-05', 'M')          # final hold-out; never used for decisions
VAL_MONTH = TEST_MONTH - 1                      # 2026-04, validation month for window selection
VAL_END = VAL_MONTH - 1                         # 2026-03, last month of validation-phase training
CANDIDATE_WINDOWS = [1, 3, 6, 12]
MIN_FREQUENCY = 5                               # ZIPs with < 5 training sales are grouped as 'infrequent'
FEATURES = mp.NUMERIC_FEATURES + mp.LOCATION_CATEGORICAL

In [2]:
first_month = mp.window_months(VAL_END, max(CANDIDATE_WINDOWS))[0]
df = mp.load_sold(min_month=first_month, max_month=TEST_MONTH)
print(f'Loaded {len(df):,} single-family rows, {df["Month"].min()} to {df["Month"].max()}.')

valid = mp.valid_rows(df)
excluded = df.loc[~valid].assign(group=lambda d: d['Month'].map(lambda m: 'test' if m == TEST_MONTH else 'pre-test'))
print(f'Feature-validity rule: LivingArea missing or < {mp.MIN_LIVING_AREA} sqft -> excluded.')
print(excluded['group'].value_counts().rename('rows_excluded').to_string())
df = df.loc[valid].reset_index(drop=True)

Loaded 153,877 single-family rows, 2025-04 to 2026-05.
Feature-validity rule: LivingArea missing or < 200 sqft -> excluded.
group
pre-test    154
test          9


## Training-window selection (validation only)

For each window length, a `LinearRegression` pipeline (log target, ZIP one-hot, structural features) is trained on the months ending 2026-03 and scored on 2026-04.
Training rows get the train-only price filter (1st–99.5th percentile of that window's `ClosePrice`); validation rows are only subject to the feature-validity rule.
The window with the lowest validation MAE is chosen. All-row validation R² is shown but is not used to choose: three April sales of $645M–$796M make up about 98% of April's total variance, so R² hardly changes across settings.
May 2026 is excluded from this cell entirely.

In [3]:
history = df.loc[df['Month'] <= VAL_MONTH]
assert TEST_MONTH not in set(history['Month'])
val_rows = history.loc[history['Month'] == VAL_MONTH]

scores = []
for window in CANDIDATE_WINDOWS:
    train_rows = history.loc[history['Month'].isin(mp.window_months(VAL_END, window))]
    train_rows, _ = mp.filter_train_prices(train_rows)
    model = mp.make_linear_model(log_target=True, log_features=[], min_frequency=MIN_FREQUENCY)
    model.fit(train_rows[FEATURES], train_rows[mp.TARGET])
    metrics = mp.regression_metrics(val_rows[mp.TARGET], model.predict(val_rows[FEATURES]))
    scores.append({'window_months': window, 'train_rows': len(train_rows), **metrics})

scores = pd.DataFrame(scores)
TRAINING_MONTHS = int(scores.sort_values('MAE').iloc[0]['window_months'])
print(scores.to_string(index=False, formatters={'R2': '{:.4f}'.format, 'MAE': '{:,.0f}'.format, 'MdAPE': '{:.2%}'.format}))
print(f'Selected TRAINING_MONTHS: {TRAINING_MONTHS}')

 window_months  train_rows     R2     MAE  MdAPE
             1       11006 0.0087 467,794 11.47%
             3       26787 0.0092 437,532 10.20%
             6       58506 0.0095 425,661  9.96%
            12      127734 0.0095 422,895  9.77%
Selected TRAINING_MONTHS: 12


In [4]:
train_months = mp.window_months(VAL_MONTH, TRAINING_MONTHS)   # final training window ends 2026-04
val_train_months = mp.window_months(VAL_END, TRAINING_MONTHS)  # validation-phase training window ends 2026-03

out = df.loc[df['Month'].isin(val_train_months.union(train_months)) | df['Month'].eq(TEST_MONTH)].copy()
out['split'] = 'history'
out.loc[out['Month'].isin(train_months), 'split'] = 'train'
out.loc[out['Month'].eq(TEST_MONTH), 'split'] = 'test'
out['val_split'] = 'none'
out.loc[out['Month'].isin(val_train_months), 'val_split'] = 'train'
out.loc[out['Month'].eq(VAL_MONTH), 'val_split'] = 'val'
out['Month'] = out['Month'].astype(str)

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
out.to_csv(OUTPUT_FILE, index=False)
print(f'Wrote {len(out):,} rows x {out.shape[1]} columns to {OUTPUT_FILE}.')
print(out.groupby('split')['Month'].agg(['min', 'max', 'count']))
print(out.groupby('val_split')['Month'].agg(['min', 'max', 'count']))

Wrote 153,714 rows x 17 columns to outputs/02_preprocessed_v2.csv.
             min      max   count
split                            
history  2025-04  2025-04   11866
test     2026-05  2026-05   12015
train    2025-05  2026-04  129833
               min      max   count
val_split                          
none       2026-05  2026-05   12015
train      2025-04  2026-03  129678
val        2026-04  2026-04   12021


In [5]:
assert set(out.loc[out['split'].eq('test'), 'Month']) == {str(TEST_MONTH)}
assert out.loc[out['split'].eq('train'), 'Month'].max() == str(VAL_MONTH)
assert out.loc[out['val_split'].eq('train'), 'Month'].max() == str(VAL_END)
assert out[mp.TARGET].notna().all() and out['LivingArea'].ge(mp.MIN_LIVING_AREA).all()
print('Checks passed.')

Checks passed.


## Preprocessing decisions

- **Row validity (all splits, including test):** rows with missing `LivingArea` or `LivingArea < 200` sqft are excluded as data-entry errors. In the training window the 0.1th percentile is about 517 sqft, so 200 sqft only catches impossible values. Test rows are never dropped because of price.
- **Training-only price filter (applied at fit time in 03/04):** training rows outside the training window's 1st–99.5th `ClosePrice` percentiles are dropped. This removes the $645M–$796M records and the sub-$1k entries.
- **Fit-on-train pipeline (`model_pipeline.py`):** numeric features are clipped to training 0.5th–99.5th percentiles, median-imputed with missing-indicator columns, and standardized. `PostalCode` is one-hot encoded, and ZIPs with fewer than 5 training sales are grouped into one infrequent bucket. ZIPs not seen in training map to that bucket as well.
- **Target:** the linear model is trained on `log(ClosePrice)`. Predictions are exponentiated back, so every metric is in dollars.
- **CSV columns:** `CloseDate`, `Month`, `ClosePrice`, the 9 numeric features, `PostalCode`, `Latitude`, `Longitude`, `split` (`train`/`test`/`history`), and `val_split` (`train`/`val`/`none`).